# Import Thư viện

In [ ]:
!pip install -q gradio

import os
import json
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import gradio as gr

from PIL import Image
from torchvision import models, transforms
from google.colab import drive

In [ ]:
# ============================================================
# 2. Mount Drive, Device, Transform và hàm tiện ích
# ============================================================

drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

def checkpoint_size_mb(path):
    return round(os.path.getsize(path) / (1024 * 1024), 2)

preprocess_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

Mounted at /content/drive
Device: cpu


# Mount Drive, Paths, Device, Load Class Mapping, Utility Functions

In [ ]:
BASE_DIR = "/content/drive/MyDrive/Deep_Learning"

CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
SPLITS_DIR = os.path.join(BASE_DIR, "data", "splits")

RESNET_PATH = os.path.join(CHECKPOINT_DIR, "resnet50_best.pth")
VIT_PATH = os.path.join(CHECKPOINT_DIR, "vit_best.pth")
CLASS_MAPPING_PATH = os.path.join(SPLITS_DIR, "class_mapping.json")

NUM_CLASSES = 43

# Kiểm tra file cần thiết
for path in [RESNET_PATH, VIT_PATH, CLASS_MAPPING_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Không tìm thấy file: {path}")

print("ResNet50 checkpoint:", RESNET_PATH)
print("ViT checkpoint:", VIT_PATH)
print("Class mapping:", CLASS_MAPPING_PATH)

# Đọc class_mapping từ Section 1 nếu cần
with open(CLASS_MAPPING_PATH, "r", encoding="utf-8") as f:
    class_mapping = json.load(f)

# Tên 43 lớp chuẩn của GTSRB
GTSRB_CLASS_NAMES = {
    0: "Speed limit (20km/h)",
    1: "Speed limit (30km/h)",
    2: "Speed limit (50km/h)",
    3: "Speed limit (60km/h)",
    4: "Speed limit (70km/h)",
    5: "Speed limit (80km/h)",
    6: "End of speed limit (80km/h)",
    7: "Speed limit (100km/h)",
    8: "Speed limit (120km/h)",
    9: "No passing",
    10: "No passing for vehicles over 3.5 tons",
    11: "Right-of-way at the next intersection",
    12: "Priority road",
    13: "Yield",
    14: "Stop",
    15: "No vehicles",
    16: "Vehicles over 3.5 tons prohibited",
    17: "No entry",
    18: "General caution",
    19: "Dangerous curve to the left",
    20: "Dangerous curve to the right",
    21: "Double curve",
    22: "Bumpy road",
    23: "Slippery road",
    24: "Road narrows on the right",
    25: "Road work",
    26: "Traffic signals",
    27: "Pedestrians",
    28: "Children crossing",
    29: "Bicycles crossing",
    30: "Beware of ice/snow",
    31: "Wild animals crossing",
    32: "End of all speed and passing limits",
    33: "Turn right ahead",
    34: "Turn left ahead",
    35: "Ahead only",
    36: "Go straight or right",
    37: "Go straight or left",
    38: "Keep right",
    39: "Keep left",
    40: "Roundabout mandatory",
    41: "End of no passing",
    42: "End of no passing by vehicles over 3.5 tons"
}

def get_sign_name(class_id):
    """
    Trả về tên biển báo từ class_id.
    Ưu tiên dùng bảng tên chuẩn GTSRB.
    """
    class_id = int(class_id)
    return GTSRB_CLASS_NAMES.get(class_id, f"Unknown class {class_id}")

print("Số lớp:", NUM_CLASSES)
print("Ví dụ class 14:", get_sign_name(14))

ResNet50 checkpoint: /content/drive/MyDrive/Deep_Learning/checkpoints/resnet50_best.pth
ViT checkpoint: /content/drive/MyDrive/Deep_Learning/checkpoints/vit_best.pth
Class mapping: /content/drive/MyDrive/Deep_Learning/data/splits/class_mapping.json
Số lớp: 43
Ví dụ class 14: Stop


# ResNet50

In [ ]:
resnet_model = models.resnet50(weights=None)

in_features = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(in_features, NUM_CLASSES)

checkpoint = torch.load(
    RESNET_PATH,
    map_location=device
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    resnet_model.load_state_dict(
        checkpoint["model_state_dict"]
    )
else:
    resnet_model.load_state_dict(checkpoint)

resnet_model.to(device)
resnet_model.eval()

print("✓ ResNet50 loaded")

✓ ResNet50 loaded


# ViT-B/16

In [ ]:
vit_model = models.vit_b_16(weights=None)

vit_model.heads.head = nn.Linear(
    vit_model.heads.head.in_features,
    NUM_CLASSES
)

checkpoint = torch.load(
    VIT_PATH,
    map_location=device
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    vit_model.load_state_dict(
        checkpoint["model_state_dict"]
    )
else:
    vit_model.load_state_dict(checkpoint)

vit_model.to(device)
vit_model.eval()

print("✓ ViT loaded")

✓ ViT loaded


# Kiểm thử thông tin model

In [ ]:
RESNET_INFO = f"""
Model: ResNet50
Checkpoint Size: {checkpoint_size_mb(RESNET_PATH)} MB
"""

VIT_INFO = f"""
Model: ViT-B/16
Checkpoint Size: {checkpoint_size_mb(VIT_PATH)} MB
"""

print(RESNET_INFO)
print(VIT_INFO)


Model: ResNet50
Checkpoint Size: 90.99 MB


Model: ViT-B/16
Checkpoint Size: 327.74 MB



# Validation/Test Transform
# Predict ResNet
# Predict ViT

In [ ]:
def predict_single_model(model, image, model_name):
    """
    Chạy inference cho một model.
    Input:
        model: ResNet50 hoặc ViT đã load checkpoint
        image: PIL Image RGB
        model_name: tên model để debug/hiển thị
    Output:
        dict chứa class_id, tên biển báo, confidence và thời gian suy luận
    """

    model.eval()

    image_tensor = preprocess_transform(image)
    image_tensor = image_tensor.unsqueeze(0).to(device)

    # Đồng bộ GPU trước khi đo thời gian
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model(image_tensor)
        probs = F.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, dim=1)

    # Đồng bộ GPU sau khi inference để đo thời gian chính xác hơn
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    inference_time = (time.perf_counter() - start) * 1000

    class_id = int(pred.item())
    confidence = float(conf.item() * 100)
    sign_name = get_sign_name(class_id)

    return {
        "model": model_name,
        "class_id": class_id,
        "sign_name": sign_name,
        "confidence": confidence,
        "time_ms": inference_time
    }


def predict_resnet(image):
    return predict_single_model(
        model=resnet_model,
        image=image,
        model_name="ResNet50"
    )


def predict_vit(image):
    return predict_single_model(
        model=vit_model,
        image=image,
        model_name="ViT-B/16"
    )

# Xây dựng Gradio

In [ ]:
def predict_image(image):
    """
    Hàm chính cho Gradio.
    Nhận ảnh từ người dùng, chạy cả ResNet50 và ViT,
    sau đó trả về kết quả dự đoán song song.
    """

    if image is None:
        return "No image uploaded.", "No image uploaded.", [], ""

    # Đảm bảo ảnh luôn là RGB 3 kênh
    image = image.convert("RGB")

    # Dự đoán bằng 2 mô hình
    resnet_result = predict_resnet(image)
    vit_result = predict_vit(image)

    # Kiểm tra 2 mô hình có dự đoán cùng class hay không
    same_prediction = (
        resnet_result["class_id"] == vit_result["class_id"]
    )

    # Lấy mô hình có confidence cao hơn trên ảnh hiện tại
    if resnet_result["confidence"] > vit_result["confidence"]:
        higher_confidence_model = "ResNet50"
        higher_confidence_result = resnet_result
    else:
        higher_confidence_model = "ViT-B/16"
        higher_confidence_result = vit_result

    # Output riêng cho ResNet50
    resnet_text = f"""
Predicted Class ID: {resnet_result['class_id']}
Traffic Sign Name: {resnet_result['sign_name']}
Confidence: {resnet_result['confidence']:.2f} %
Inference Time: {resnet_result['time_ms']:.2f} ms
"""

    # Output riêng cho ViT
    vit_text = f"""
Predicted Class ID: {vit_result['class_id']}
Traffic Sign Name: {vit_result['sign_name']}
Confidence: {vit_result['confidence']:.2f} %
Inference Time: {vit_result['time_ms']:.2f} ms
"""

    # Bảng so sánh 2 mô hình
    comparison = [
        [
            "ResNet50",
            resnet_result["class_id"],
            resnet_result["sign_name"],
            round(resnet_result["confidence"], 2),
            round(resnet_result["time_ms"], 2)
        ],
        [
            "ViT-B/16",
            vit_result["class_id"],
            vit_result["sign_name"],
            round(vit_result["confidence"], 2),
            round(vit_result["time_ms"], 2)
        ]
    ]

    # Ghi chú so sánh
    note = f"""
Hai mô hình dự đoán giống nhau: {"CÓ" if same_prediction else "KHÔNG"}

Mô hình có độ tin cậy cao hơn trên ảnh này: {higher_confidence_model}

Class ID dự đoán: {higher_confidence_result["class_id"]}
Tên biển báo: {higher_confidence_result["sign_name"]}
Độ tin cậy: {higher_confidence_result["confidence"]:.2f} %
"""

    return resnet_text, vit_text, comparison, note


# Input ảnh
image_input = gr.Image(
    type="pil",
    label="Upload Traffic Sign Image"
)

# Output ResNet50
resnet_output = gr.Textbox(
    label="ResNet50 Prediction",
    lines=5
)

# Output ViT
vit_output = gr.Textbox(
    label="ViT-B/16 Prediction",
    lines=5
)

# Bảng so sánh
comparison_table = gr.Dataframe(
    headers=[
        "Model",
        "Class ID",
        "Traffic Sign Name",
        "Confidence (%)",
        "Inference Time (ms)"
    ],
    label="Model Comparison"
)

# Ghi chú so sánh
note_output = gr.Textbox(
    label="Comparison Note",
    lines=6
)

# Giao diện Gradio
demo = gr.Interface(
    fn=predict_image,
    inputs=image_input,
    outputs=[
        resnet_output,
        vit_output,
        comparison_table,
        note_output
    ],
    title="GTSRB Traffic Sign Classification Demo",
    description=(
        "Upload a traffic sign image. "
        "The system will compare predictions from ResNet50 and ViT-B/16."
    )
)

demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e4d9fdb71bac1073c8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
